
# 🧪 SageMaker Pipeline — Ready-to-Run for `mlops-zoomcamp-project-cohort-2024`

This notebook builds a **minimal MLOps pipeline** on SageMaker using your repository data:

**Pipeline steps**
1) **Process** raw CSV → train/test split  
2) **Train** a simple model (LogisticRegression for classification or LinearRegression for regression—here we use LinearRegression for salary)  
3) **Evaluate** on test set → produce `metrics.json`  
4) **Register** the model in SageMaker Model Registry  
5) *(Optional)* **Deploy** to a test endpoint

> Works best in **Amazon SageMaker Studio / Studio Classic** with an execution role that has access to S3/ECR/CloudWatch.


In [1]:

# If running in SageMaker Studio, the following should work out of the box.
# If not, set AWS credentials/role appropriately.
!pip -q install "sagemaker==2.*" boto3 botocore


In [1]:

import os, json, boto3, sagemaker, pandas as pd
from sagemaker import get_execution_role
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.parameters import ParameterString, ParameterInteger
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.functions import Join

# 👇 use the SKLearn wrappers instead of image_uris
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn

session = sagemaker.Session()
region = session.boto_region_name

try:
    role = get_execution_role()
except Exception:
    role = os.environ.get("SM_EXECUTION_ROLE", "")

print("Region:", region)
print("Role:", role)




sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: ap-south-1
Role: arn:aws:iam::911167910451:role/service-role/AmazonSageMaker-ExecutionRole-20251031T001712


In [2]:

# ==== USER PARAMS (edit if you like) ====
# A globally-unique bucket is required. If you already have one, set it below and set CREATE_BUCKET=False.
PROJECT_NAME = "mlops-zoomcamp-poc"
BUCKET = os.environ.get("SM_PIPELINE_BUCKET", f"sagemaker-{region}-{boto3.client('sts').get_caller_identity()['Account']}")
PREFIX = f"{PROJECT_NAME}"
CREATE_BUCKET = False  # set True if you want to create a new bucket

# Pipeline/config params
PIPELINE_NAME = "MLOpsZoomcampPOC"
MODEL_PACKAGE_GROUP = "MLOpsZoomcampPOCGroup"
INSTANCE_TYPE_PROCESS = "ml.m5.large"
INSTANCE_TYPE_TRAIN = "ml.m5.large"


In [3]:

s3 = boto3.resource("s3")
if CREATE_BUCKET:
    try:
        if region == "us-east-1":
            s3.create_bucket(Bucket=BUCKET)
        else:
            s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": region})
        print("Created bucket:", BUCKET)
    except Exception as e:
        print("Bucket may already exist or not owned by you:", e)

print("Using bucket:", BUCKET)


Using bucket: sagemaker-ap-south-1-911167910451


In [5]:

# Clone your repo (public) and upload the CSV to S3 for the pipeline
REPO_URL = "https://github.com/nilarte/mlops-zoomcamp-project-cohort-2024"
LOCAL_REPO = "mlops-zoomcamp-project-cohort-2024"

if not os.path.exists(LOCAL_REPO):
    !git clone {REPO_URL} {LOCAL_REPO}

csv_path = os.path.join(LOCAL_REPO, "employee_data.csv")
assert os.path.exists(csv_path), "employee_data.csv not found in repo root"

input_s3_uri = f"s3://{BUCKET}/{PREFIX}/data/employee_data.csv"
!aws s3 cp {csv_path} {input_s3_uri}
print("Uploaded data to:", input_s3_uri)


upload: mlops-zoomcamp-project-cohort-2024/employee_data.csv to s3://sagemaker-ap-south-1-911167910451/mlops-zoomcamp-poc/data/employee_data.csv
Uploaded data to: s3://sagemaker-ap-south-1-911167910451/mlops-zoomcamp-poc/data/employee_data.csv


In [4]:

# Create processing script (train/test split). Saves to /opt/ml/processing/{train,test}
processing_script = """
import os
import pandas as pd
from sklearn.model_selection import train_test_split

import argparse
parser = argparse.ArgumentParser()
parser.add_argument("--input_csv", type=str, required=True)
args = parser.parse_args()

src = args.input_csv
print(f"🔎 Reading input file: {src}")

#df = pd.read_csv(args.input_csv)

# 1) read safely
try:
    df = pd.read_csv(src)
except Exception as e:
    print("⚠️ default read_csv failed, trying engine='python' and comma sep")
    df = pd.read_csv(src, engine="python", sep=",")
    # if still bad, raise
    # (you can add sniffing here)

print("✅ Read shape:", df.shape)


# Very basic cleaning: drop NA
df = df.dropna()

# Save raw columns layout; assume last column is target ("Salary" in this dataset)
# If the last column isn't Salary in your repo, adjust here.
target_col = df.columns[-1]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

os.makedirs("/opt/ml/processing/train", exist_ok=True)
os.makedirs("/opt/ml/processing/test", exist_ok=True)

train_df.to_csv("/opt/ml/processing/train/train.csv", index=False)
test_df.to_csv("/opt/ml/processing/test/test.csv", index=False)
print("✅ Processing complete. Rows -> train:", len(train_df), " test:", len(test_df))
"""

with open("processing.py", "w") as f:
    f.write(processing_script)

print("Wrote processing.py")


Wrote processing.py


In [24]:

# Create simple training script (Linear Regression for salary prediction)
train_script = """
import os, json, argparse
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import joblib

parser = argparse.ArgumentParser()
parser.add_argument("--train_csv", type=str, default="/opt/ml/input/data/train/train.csv")
parser.add_argument("--test_csv", type=str, default="/opt/ml/input/data/test/test.csv")
args = parser.parse_args()
args = parser.parse_args()

train = pd.read_csv(args.train_csv)
test = pd.read_csv(args.test_csv)

# Assume last column is target
X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = mean_squared_error(y_test, preds, squared=False)

# Save model
model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, "model.joblib"))

# Save metrics.json for the pipeline to read
metrics = {"rmse": float(rmse)}
os.makedirs("/opt/ml/output", exist_ok=True)
with open("/opt/ml/output/metrics.json", "w") as f:
    json.dump(metrics, f)

print("✅ Training complete. RMSE:", rmse)
"""

with open("train.py", "w") as f:
    f.write(train_script)

print("Wrote train.py")


Wrote train.py


In [25]:

from sagemaker.image_uris import retrieve as image_uri_retrieve


# Parameters
processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value=INSTANCE_TYPE_PROCESS)
training_instance_type   = ParameterString(name="TrainingInstanceType",   default_value=INSTANCE_TYPE_TRAIN)
train_instance_count     = ParameterInteger(name="TrainInstanceCount", default_value=1)

# Processing Step
processing_instance_type = ParameterString(name="ProcessingInstanceType", default_value="ml.m5.large")
training_instance_type   = ParameterString(name="TrainingInstanceType",   default_value="ml.m5.large")
train_instance_count     = ParameterInteger(name="TrainInstanceCount", default_value=1)

input_s3_uri = f"s3://{BUCKET}/{PREFIX}/data/employee_data.csv"

processor = SKLearnProcessor(
    framework_version="1.2-1",      # pick from the versions your error listed
    role=role,
    instance_type=processing_instance_type.default_value,
    instance_count=1,
)

process_step = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    code="processing.py",           # the script we wrote earlier
    job_arguments=["--input_csv", "/opt/ml/processing/input/employee_data.csv"],
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=input_s3_uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        sagemaker.processing.ProcessingOutput(output_name="test",  source="/opt/ml/processing/test"),
    ],
)


# Training Step
sk_estimator = SKLearn(
    entry_point="train.py",          # the training script we wrote
    framework_version="1.2-1",       # same as above
    role=role,
    instance_type=training_instance_type.default_value,
    instance_count=train_instance_count.default_value,
    base_job_name="mlops-zoomcamp-train",
    output_path=f"s3://{BUCKET}/{PREFIX}/model-artifacts/",
)

train_step = TrainingStep(
    name="TrainModel",
    estimator=sk_estimator,
    inputs={
        "train": process_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
        "test":  process_step.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
    },
)



# Register Model Step
register_step = RegisterModel(
    name="RegisterModel",
    estimator=sk_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    model_package_group_name="MLOpsZoomcampPOCGroup",
    content_types=["text/csv"],          # 👈 required in your SDK
    response_types=["text/csv"],         # 👈 required in your SDK
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
)



pipeline = Pipeline(
    name=PIPELINE_NAME,
    steps=[process_step, train_step, register_step],
    sagemaker_session=session,
)

print("Pipeline definition is ready.")


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Pipeline definition is ready.


In [26]:

# Upsert (create or update) and start an execution
pipeline.upsert(role_arn=role)
execution = pipeline.start()
execution


_PipelineExecution(arn='arn:aws:sagemaker:ap-south-1:911167910451:pipeline/MLOpsZoomcampPOC/execution/hu4p7f1w9odr', sagemaker_session=<sagemaker.session.Session object at 0x7f099554fb90>)

In [ ]:

# (Optional) Wait for completion here. You can also watch progress in the SageMaker Console → Pipelines.
execution.wait()
print("Pipeline execution completed:", execution.describe()['PipelineExecutionStatus'])


In [ ]:

# === OPTIONAL: Deploy the latest approved model as a serverless endpoint ===
# You can skip this if you only need the registry entry.
from sagemaker import ModelPackage
from time import strftime

endpoint_name = f"{PROJECT_NAME}-ep-{strftime('%Y%m%d%H%M%S')}"

# Find the latest model package in the group
sm = boto3.client("sagemaker")
resp = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)
if resp["ModelPackageSummaryList"]:
    model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
    print("Latest model package:", model_package_arn)

    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=session
    )
    # Use Serverless Inference for low-cost demo
    predictor = model.deploy(
        serverless_inference_config={"MemorySizeInMB": 2048, "MaxConcurrency": 1},
        endpoint_name=endpoint_name,
    )
    print("✅ Deployed endpoint:", endpoint_name)
else:
    print("No model packages found to deploy yet.")


In [ ]:

# === OPTIONAL: Invoke the endpoint with one sample ===
# We'll read one row from the test split and call predict.
import io, numpy as np

test_df = pd.read_csv(f"s3://{BUCKET}/{PREFIX}/data/employee_data.csv")
# Toy example: take one row and drop the target (assumed last column)
if test_df.shape[0] > 0:
    sample = test_df.iloc[[0], :-1]
    csv_payload = sample.to_csv(header=False, index=False)
    try:
        response = predictor.predict(csv_payload)
        print("Prediction response:", response)
    except Exception as e:
        print("Prediction skipped or endpoint not deployed:", e)
